# ETF Tricks Notebook Quickstart

## Goal

以單一 `ETFTrickLab` 入口產出 13 個台股 ETF Tricks 的 Daily NAV、Daily ETF 成交金額與可稽核明細，並示範 FFD 輸出介面及任意資金的實際整股配置。Notebook 不重複核心公式。

## Setup

請使用 repository-local `.venv` kernel。調整日期與資金後由上至下執行。

In [ ]:
from decimal import Decimal
from pathlib import Path

from etf_tricks import ETFTrickLab

DATA_ANALYSTS_ROOT = Path.cwd() / "DataAnalysts"
START_DATE = "2026-07-01"
END_DATE = "2026-07-07"
INITIAL_CAPITAL = Decimal("10000000")

## Steps

### 1. 產出 13 個 ETF Tricks

In [ ]:
lab = ETFTrickLab.from_data_analysts(DATA_ANALYSTS_ROOT)
result = lab.run_all(
    start_date=START_DATE,
    end_date=END_DATE,
    initial_capital=INITIAL_CAPITAL,
)

### 2. 檢視 Daily 曲線與稽核表

In [ ]:
display(result.nav.tail())
display(result.amount.tail())
display(result.holdings.tail())
display(result.trades.tail())
display(result.targets.tail())
display(result.candidates.tail())

### 3. 匯出未來 FFD 所需的薄介面

此步只輸出欄位，不執行 FFD，也不選擇 `d*`。

In [ ]:
ffd_input = result.for_ffd("momentum")
display(ffd_input.tail())

### 4. 把任意資金拆成實際股票與整股數

In [ ]:
latest_formation_date = result.targets["formation_date"].max()
allocation = lab.allocate(
    etf_id="momentum",
    as_of_date=latest_formation_date,
    capital=Decimal("25000000"),
)
display(allocation.basket)
display(allocation.orders)
display(allocation.schedule)

## Checks

In [ ]:
readiness = lab.validate(result)
print(readiness.status)
display(readiness.per_etf)
display(readiness.目前可用)
display(readiness.目前缺失限制)

## Next Steps

只有 readiness 為 `READY` 時，才將 `for_ffd()` 的完整歷史輸出交給未來的 Dollar bar / FFD 工作。FFD、ADF 與 ML 不在本 Notebook 的處理範圍。